# 📄 LLM File System Assistant — Project Workbook

This notebook demonstrates a complete end-to-end implementation of an **LLM-powered File System Assistant** that uses **function calling / tool use** to interact with resume documents.

---

## 🗂️ Project Overview

| Component | Description |
|---|---|
| **Part A** | Core File System Tools (`read_file`, `list_files`, `write_file`, `search_in_file`) |
| **Part B** | LLM Integration with function calling (OpenAI / OpenRouter API) |

### Learning Objectives
- Understand LLM function calling / tool use patterns
- Implement structured tool interfaces with JSON schemas
- Handle file I/O operations programmatically (PDF, DOCX, TXT)
- Parse and validate documents

### Files in the Project
```
llm-file-system-assistant/
├── fs_tools.py             # Core file system tools (Part A)
├── llm_file_assistant.py   # LLM integration & function calling (Part B)
├── requirements.txt        # Dependencies
├── .env                    # API key configuration
├── README.md               # Project documentation
├── workbook.ipynb          # This notebook
└── resumes/                # Sample PDF resume files

---
##  Dependencies Setup

All required Python packages (`openai`, `python-dotenv`, `pypdf`, `python-docx`, `reportlab`) are specified in `requirements.txt`. Ensure they are installed in your environment before running this notebook:

```bash
pip install -r requirements.txt
```

---
## Configure API Keys

Load environment variables from the `.env` file. 
Ensure your `.env` contains one of the following:
```ini
OPENROUTER_API_KEY=sk-or-v1-...
# OR
OPENAI_API_KEY=sk-...
```

In [2]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

OPENROUTER_KEY = os.environ.get("OPENROUTER_API_KEY")
OPENAI_KEY = os.environ.get("OPENAI_API_KEY")

if OPENROUTER_KEY:
    print("Using: OpenRouter API")
    client = OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=OPENROUTER_KEY
    )
    MODEL = os.environ.get("MODEL_NAME", "openai/gpt-4o-mini")
elif OPENAI_KEY:
    print("Using: OpenAI API")
    client = OpenAI(api_key=OPENAI_KEY)
    MODEL = os.environ.get("MODEL_NAME", "gpt-4o-mini")
else:
    raise EnvironmentError("No API key found. Please set OPENROUTER_API_KEY or OPENAI_API_KEY in your .env file.")

print(f"Model: {MODEL}")

Using: OpenRouter API
Model: openai/gpt-4o-mini


---
## 🛠️ Part A — Core File System Tools

Define the four core tool functions that will be exposed to the LLM. Each returns structured dictionaries for predictable, machine-readable responses.

### Tool 1: `read_file(filepath)` — Read Resume Documents

Reads a file (`.pdf`, `.txt`, `.docx`) and extracts:
- Full text content
- Metadata: name, path, size, type, modification date
- Graceful error handling for missing or unsupported files

In [8]:
%pip install python-docx

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
import datetime
from pathlib import Path

# PDF parsing
from pypdf import PdfReader

# DOCX parsing
import docx

In [10]:
def read_file(filepath: str) -> dict:
    """
    Read a resume file (PDF, TXT, DOCX) and extract text content along with metadata.

    Args:
        filepath (str): Path to the file to read.

    Returns:
        dict: {'status', 'content', 'metadata'} on success,
              {'status', 'error'} on failure.
    """
    try:
        path = Path(filepath)
        if not path.exists():
            return {"status": "failed", "error": f"File not found: {filepath}"}
        if not path.is_file():
            return {"status": "failed", "error": f"Path is not a file: {filepath}"}

        extension = path.suffix.lower()
        content = ""

        if extension == ".txt":
            with open(path, "r", encoding="utf-8", errors="ignore") as f:
                content = f.read()
        elif extension == ".pdf":
            reader = PdfReader(str(path))
            content = "\n".join(
                page.extract_text() for page in reader.pages if page.extract_text()
            )
        elif extension == ".docx":
            doc_obj = docx.Document(str(path))
            content = "\n".join(p.text for p in doc_obj.paragraphs)
        else:
            return {
                "status": "failed",
                "error": f"Unsupported file type: '{extension}'. Supported: .pdf, .txt, .docx"
            }

        stat = path.stat()
        mod_time = datetime.datetime.fromtimestamp(
            stat.st_mtime, tz=datetime.timezone.utc
        ).strftime("%Y-%m-%d %H:%M:%S UTC")

        return {
            "status": "success",
            "content": content,
            "metadata": {
                "name": path.name,
                "filepath": str(path),
                "size_bytes": stat.st_size,
                "type": extension,
                "modified_date": mod_time
            }
        }
    except Exception as e:
        return {"status": "failed", "error": str(e)}


# --- Demo: Read a PDF resume ---
result = read_file("resumes/resume_john_doe.pdf")
print(f"Status  : {result['status']}")
print(f"File    : {result['metadata']['name']}")
print(f"Size    : {result['metadata']['size_bytes']} bytes")
print(f"Modified: {result['metadata']['modified_date']}")
print(f"\nContent Preview (first 300 chars):\n{result['content'][:300]}")

Status  : success
File    : resume_john_doe.pdf
Size    : 2239 bytes
Modified: 2026-07-23 06:44:40 UTC

Content Preview (first 300 chars):
John Doe - Senior Software Engineer
Professional Summary:
Experienced Software Engineer specializing in backend infrastructure, REST API design, and
distributed systems. Expert in Python, FastAPI, and Docker.
Technical Skills:
Python, FastAPI, Docker, PostgreSQL, AWS, Redis, Git
Work Experience:
 S


### Tool 2: `list_files(directory, extension)` — List Directory Contents

Lists all files in a directory with:
- Name, path, size, modified date, type
- Optional extension filter (e.g. `.pdf`, `.txt`)

In [11]:
def list_files(directory: str, extension: str = None) -> list:
    """
    List all files in a directory, optionally filtered by extension.

    Args:
        directory (str): Directory path to list files from.
        extension (str, optional): Extension filter e.g. '.pdf' or 'pdf'.

    Returns:
        list: List of file metadata dicts, or error dict in a list.
    """
    try:
        path = Path(directory)
        if not path.exists() or not path.is_dir():
            return [{"status": "failed", "error": f"Directory not found: {directory}"}]

        ext = None
        if extension:
            ext = extension.strip().lower()
            if not ext.startswith("."):
                ext = f".{ext}"

        files = []
        for item in sorted(path.iterdir()):
            if item.is_file():
                if ext and item.suffix.lower() != ext:
                    continue
                stat = item.stat()
                mod_time = datetime.datetime.fromtimestamp(
                    stat.st_mtime, tz=datetime.timezone.utc
                ).strftime("%Y-%m-%d %H:%M:%S UTC")
                files.append({
                    "name": item.name,
                    "path": str(item),
                    "size_bytes": stat.st_size,
                    "modified_date": mod_time,
                    "type": item.suffix.lower()
                })
        return files
    except Exception as e:
        return [{"status": "failed", "error": str(e)}]


# --- Demo: List all resumes ---
files = list_files("resumes")
print(f"Found {len(files)} resume files:\n")
for f in files:
    print(f"  {f['name']:35s}  {f['size_bytes']:>6} bytes   {f['modified_date']}")

Found 10 resume files:

  resume_alice_harper.pdf                2144 bytes   2026-07-23 06:27:42 UTC
  resume_bob_williams.pdf                2122 bytes   2026-07-23 06:44:40 UTC
  resume_clara_bennett.pdf               2126 bytes   2026-07-23 06:27:42 UTC
  resume_david_miller.pdf                2086 bytes   2026-07-23 06:44:40 UTC
  resume_elena_rostova.pdf               2113 bytes   2026-07-23 06:27:42 UTC
  resume_eva_davis.pdf                   2146 bytes   2026-07-23 06:44:40 UTC
  resume_jane_smith.pdf                  2208 bytes   2026-07-23 06:44:40 UTC
  resume_john_doe.pdf                    2239 bytes   2026-07-23 06:44:40 UTC
  resume_marcus_vane.pdf                 2129 bytes   2026-07-23 06:27:42 UTC
  resume_samuel_brooks.pdf               2116 bytes   2026-07-23 06:27:42 UTC


### Tool 3: `write_file(filepath, content)` — Write Files to Disk

Writes text content to any file path:
- Creates parent directories automatically
- Returns success status, path, and file size

In [12]:
def write_file(filepath: str, content: str) -> dict:
    """
    Write text content to a file, creating parent directories if necessary.

    Args:
        filepath (str): Destination file path.
        content (str): Text content to write.

    Returns:
        dict: {'status', 'message', 'filepath', 'size_bytes'} on success.
    """
    try:
        path = Path(filepath)
        path.parent.mkdir(parents=True, exist_ok=True)
        with open(path, "w", encoding="utf-8") as f:
            f.write(content)
        stat = path.stat()
        return {
            "status": "success",
            "message": f"Successfully written to {filepath}",
            "filepath": str(path),
            "size_bytes": stat.st_size
        }
    except Exception as e:
        return {"status": "failed", "error": str(e)}


# --- Demo: Write a summary file ---
sample_content = """RESUME SUMMARY — John Doe
Role   : Senior Software Engineer
Skills : Python, FastAPI, Docker, AWS, PostgreSQL
Status : Strong Python background — recommend for interview.
"""

result = write_file("summaries/john_doe_summary.txt", sample_content)
print(f"Status  : {result['status']}")
print(f"Written : {result['filepath']}")
print(f"Size    : {result['size_bytes']} bytes")

Status  : success
Written : summaries\john_doe_summary.txt
Size    : 179 bytes


### Tool 4: `search_in_file(filepath, keyword)` — Keyword Search with Context

Searches a document for a keyword (case-insensitive) and returns:
- Line number of each match
- The matched line
- Surrounding context (1 line before and after)

In [13]:
def search_in_file(filepath: str, keyword: str) -> dict:
    """
    Search for a keyword in a file (case-insensitive) with surrounding context.

    Args:
        filepath (str): File path to search in.
        keyword (str): Keyword or phrase to search for.

    Returns:
        dict: {'status', 'keyword', 'matches_found', 'matches'} on success.
    """
    try:
        read_result = read_file(filepath)
        if read_result.get("status") == "failed":
            return read_result

        content = read_result.get("content", "")
        lines = content.splitlines()
        keyword_lower = keyword.lower()

        matches = []
        for i, line in enumerate(lines):
            if keyword_lower in line.lower():
                start = max(0, i - 1)
                end = min(len(lines), i + 2)
                matches.append({
                    "line_number": i + 1,
                    "match": line.strip(),
                    "context": "\n".join(lines[start:end])
                })

        return {
            "status": "success",
            "filepath": str(filepath),
            "keyword": keyword,
            "matches_found": len(matches),
            "matches": matches
        }
    except Exception as e:
        return {"status": "failed", "error": str(e)}


# --- Demo: Search for 'Python' in John Doe's resume ---
result = search_in_file("resumes/resume_john_doe.pdf", "Python")
print(f"Keyword      : '{result['keyword']}'")
print(f"File         : {result['filepath']}")
print(f"Matches Found: {result['matches_found']}\n")

for m in result["matches"]:
    print(f"  Line {m['line_number']}: {m['match']}")

Keyword      : 'Python'
File         : resumes/resume_john_doe.pdf
Matches Found: 5

  Line 4: distributed systems. Expert in Python, FastAPI, and Docker.
  Line 6: Python, FastAPI, Docker, PostgreSQL, AWS, Redis, Git
  Line 8:  Senior Python Developer - TechCorp Innovations (2021 - Present)
  Line 9: Architected high-throughput microservices using Python and FastAPI. Reduced API latency by 40%
  Line 12: Built containerized data pipelines with Python and Docker on AWS ECS.


---
## 🤖 Part B — LLM Function Calling Integration

Define the OpenAI-compatible JSON schemas for each tool and wire them into an LLM-driven agentic loop.
The LLM autonomously decides **which tools to call, in what order, and with what arguments** based on a user query.